<a href="https://colab.research.google.com/github/3160664535qqcom-ctrl/kangtai-AnalyticsEngineering/blob/main/model_answer_testing_and_git.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# MODEL ANSWER: Testing and Git: From notebook to trustworthy code

| Part | Time | Work in | What you do |
| --- | --- | --- | --- |
| 1. Testing | 30 min | pairs/small groups | Write tests for small pipeline functions, then write data checks for a silver table |
| 2. Version control | 15 min | individually | Run git from the notebook: commit, inspect a change, find a bug and undo it |

**Why this matters.** In the lecture we listed what goes wrong when analytics code lives only in a notebook: it is run in the wrong order, it has hidden assumptions, and nobody knows what changed. Two practices address most of that:

- **Tests** turn "I think this works" into evidence that is repeatable and can be run by a machine.
- **Version control** records every change, who made it and why, so a mistake can be found and undone.

You will use the card-spend data from the pipeline practice task, so there is no new dataset to learn.

**Two things to know before you start**
- You are not writing a pipeline here. The functions are provided; your job is to decide **what they should do** and prove it.
- Part 2 uses git commands typed into notebook cells with a leading `!`. Git is a tool installed on the Colab machine; GitHub is a website that stores a copy of a repository. Here you use git only.

In [ ]:
# ---- Setup: run this cell first ----
!pip install -q pytest

import glob
import os
import subprocess

import pandas as pd

# Instructor: set DATA_URL to the link of the card-spend practice zip.
DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/testgitsession/card-spend-practice.zip"
DATA_DIR = "/content/card-spend"

# Download only if no CSV files are there yet. The CSVs may sit inside a folder within the zip.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/card-spend.zip")
    zipfile.ZipFile("/content/card-spend.zip").extractall(DATA_DIR)


def find(filename):
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if not found:
        have = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
        raise FileNotFoundError(f"Missing {filename}. CSV files found under {DATA_DIR}: {have}. Ask for help.")
    return found[0]


bronze_tx = pd.read_csv(find("card_transactions.csv"), dtype={"ref_txn_id": "string"})
bronze_cu = pd.read_csv(find("customer_master.csv"))

WORK = os.getcwd()   # the folder where this notebook works; Part 2 returns to it
print(f"bronze_tx: {bronze_tx.shape[0]:,} rows | bronze_cu: {bronze_cu.shape[0]:,} rows | working folder: {WORK}")

---
# Part 1. Testing

## 1.1 What a test is

A test is a small piece of code that calls a function with an input you chose and checks the result against what you expect. If the result is different, the test fails and tells you.

That sounds trivial, but it forces a decision that is easy to skip: **what should this function do?** In a notebook that decision stays in your head. In a test it is written down.

Run the next cell. It shows the raw currency codes in the bronze data.

In [ ]:
%cd {WORK}
bronze_tx["currency"].value_counts()

Lower-case codes and trailing spaces. A function that cleans these should be *proved* to work, not assumed to.

Below is the module with the functions you will test. **Read it, but do not edit it.** It is written to a file, `pipeline.py`, because that is how real code is shared: one file, many callers.

In [ ]:
%%writefile pipeline.py
"""The above magic function ("%%writefile") means we are creating a .py script in this cell"""

"""Small, pure functions for the card-spend pipeline (silver layer)."""
import pandas as pd

# Later statuses outrank earlier ones when two rows share a file date.
RANK = {"PENDING": 0, "POSTED": 1, "REVERSED": 2}


def clean_currency(series):
    """
    Strip spaces and upper-case currency codes.
    INPUT: a Series of data
    OUTPUT: Returns a new Series.
    """
    return series.str.strip().str.upper()


def parse_timestamp(text):
    """
    Read a processor timestamp. Two formats arrive:
    ISO 'YYYY-MM-DD HH:MM:SS' and day-first 'DD/MM/YYYY HH:MM' (contains a '/').
    Anything else raises ValueError.
    INPUT: A date value as string
    OUTPUT: A date value as datetime (or an error)
    """
    fmt = "%d/%m/%Y %H:%M" if "/" in text else "%Y-%m-%d %H:%M:%S"
    return pd.to_datetime(text, format=fmt)


def latest_status(df):
    """
    Return one row per txn_id: the row from the latest file_date.
    If two rows share a file_date, the higher-ranked status wins (see RANK).
    Exact duplicate rows are removed first. The input is not changed.
    INPUT: A dataframe
    OUTPUT: A modified dataframe
    """
    d = df.drop_duplicates().copy()
    d["_rank"] = d["status"].map(RANK)
    d = d.sort_values(["txn_id", "file_date", "_rank"])
    return d.groupby("txn_id").tail(1).drop(columns="_rank").reset_index(drop=True)


def gbp_amount(amount, currency, rates):
    """
    Convert an amount to GBP. `rates` maps currency -> GBP value of one unit.
    GBP is returned unchanged. An unknown currency raises KeyError (it is never guessed).
    INPUT: Amount, currency, rates
    OUTPUT: Amount in GBP
    """
    if currency == "GBP":
        return amount
    return amount * rates[currency]


def build_silver(bronze):
    """
    Bronze transactions -> silver: duplicates removed, codes cleaned, timestamps parsed, latest status kept.
    INPUT: A dataframe (bronze)
    OUTPUT: A modified dataframe (silver)
    """
    d = bronze.drop_duplicates().copy()
    d["currency"] = clean_currency(d["currency"])
    d["ts"] = d["txn_timestamp"].map(parse_timestamp)
    return latest_status(d)

## 1.2 Write the unit tests

The next cell writes a second file, `tests.py`. A test is a function whose name starts with `test_`, containing `assert` statements. An `assert` passes silently if the condition is true and fails loudly if not.

- The first test is done for you as a **worked example**.
- For each remaining test, the docstring tells you the **situation**. You decide **what the correct answer is** and write the `assert`.
- Build small tables with the helper `make_rows(...)`. You do not need the real data for a unit test; tiny, hand-made inputs are the point.
- Aim to finish **`parse_timestamp` and `latest_status`**. `gbp_amount` is the stretch.

Unfinished tests raise `NotImplementedError`, so they show as **failed** until you write them. Re-run the cell whenever you change it.

*Hint: `pytest.raises(ValueError)` checks that code raises an error, and `pytest.approx(x)` compares decimals safely.*

### **NOTE:**
This is the model answer so tests are filled in below. There are many ways to implement these so if your's is different do not worry. If it works, it works :)

In [ ]:
%%writefile tests.py
import pandas as pd
import pytest

from pipeline import clean_currency, parse_timestamp, latest_status, gbp_amount


def make_rows(*rows):
    """Build a small bronze table. Each row is (txn_id, status, file_date, amount)."""
    return pd.DataFrame(rows, columns=["txn_id", "status", "file_date", "amount"])


# ---------------- worked example ----------------
def test_clean_currency_fixes_case_and_spaces():
    raw = pd.Series(["gbp", "EUR ", " usd", "GBP"])
    assert clean_currency(raw).tolist() == ["GBP", "EUR", "USD", "GBP"]

'''
COMMENTARY ON THE EXAMPLE
We can see that the raw sequence ["gbp", "EUR ", " usd", "GBP"] would fail if there
were no function as they do not match the expected output - ["GBP", "EUR", "USD", "GBP"].
We call the clean_currency function on raw and it should change it to upper case and remove the spaces.
This should mean we pass the test now.
The 'assert' commands basically means test whether the function output matches the expected output.
The test should pass
'''


# ---------------- parse_timestamp ----------------
def test_parse_iso():
    # as we did in class
    assert parse_timestamp("2026-03-08 13:25:00") == pd.Timestamp("2026-03-08 13:25")


def test_parse_day_first():
    # Day first: 8 March, not 3 August.
    # If it works the date object should be 2026-03-08 not 2026-08-03
    assert parse_timestamp("08/03/2026 13:25") == pd.Timestamp("2026-03-08 13:25")


# ---------------- latest_status ----------------
def test_later_file_wins():
    # maket the dataframe
    df = make_rows(("T1", "PENDING", "2026-01-05", 10.0), ("T1", "POSTED", "2026-01-06", 10.0))
    out = latest_status(df) # run the function
    assert len(out) == 1 # should only be one kept
    assert out.loc[0, "status"] == "POSTED" # and it should be posted


def test_same_file_prefers_posted():
    # maket the dataframe
    df = make_rows(("T1", "POSTED", "2026-01-05", 10.0), ("T1", "PENDING", "2026-01-05", 10.0))
    assert latest_status(df).loc[0, "status"] == "POSTED" # answer should be posted as the one kept


def test_exact_duplicates_are_removed():
    # maket the dataframe - two duplicates
    df = make_rows(("T1", "POSTED", "2026-01-05", 10.0), ("T1", "POSTED", "2026-01-05", 10.0))
    # after the function the exact duplicates should be seen and only one kept
    assert len(latest_status(df)) == 1 # i.e. the length should be one


# ---------------- gbp_amount (stretch - if you have time) ----------------
def test_gbp_is_unchanged():
    # as the example below is currency == "GBP" the amount should not change
    assert gbp_amount(10.0, "GBP", {"EUR": 0.85}) == 10.0


def test_eur_is_converted():
    # 100 euros at a rate of 0.85 should be 85 pounds
    assert gbp_amount(100.0, "EUR", {"EUR": 0.85}) == pytest.approx(85.0) # use approx in case of rounding


def test_unknown_currency_is_not_guessed():
    # Silently assuming a rate (or 1.0) would put wrong numbers in the report with no warning.
    with pytest.raises(KeyError): # Japanese Yen is not included so should be an error not converted.
        gbp_amount(5.0, "JPY", {"EUR": 0.85})

Overwriting tests.py


Now run the tests. `-q` keeps the output short.

In [ ]:
!python -m pytest -q tests.py

.........                                                                [100%]
9 passed in 0.50s


Read the result line at the bottom (for example `3 failed, 7 passed`). If a test fails, read the message before changing anything: *is the test wrong, or is the function wrong?* Either is possible. Deciding which is the real skill.

**Checkpoint.** You should have at least the `parse_timestamp` and `test_parse_iso` tests passing. Keep your `tests.py` as it is: you will use it in Part 2.

## 1.3 Data tests on a silver table

Unit tests check functions on tiny inputs. Data tests check the actual table a pipeline has produced. They catch problems that no one thought to put in a unit test, such as a new kind of bad row arriving next month.

The standard data tests have names you will meet again in the module and in practice: **unique**, **not null**, **accepted values** and **relationships**.

Every check needs a decision as well as a condition: *what happens when it fails?*

| Action | Meaning |
| --- | --- |
| `fail` | Stop the pipeline. The problem is serious enough that nothing downstream can be trusted. |
| `quarantine` | Move the bad rows aside for someone to review; carry on with the rest. |
| `flag` | Keep the rows, mark them, and carry on. |

First build the silver table with the provided function:

In [ ]:
import importlib
import sys

# Add the working folder to the front of Python's module search path,
# so "import pipeline" finds the pipeline.py file that was written there.
# WORK was defined in the setup cell (the folder the notebook works in).
sys.path.insert(0, WORK)

import pipeline    # load pipeline.py as a module, so its functions can be called as pipeline.<name>

# Python remembers modules it has already imported. If pipeline.py has been
# rewritten since the first import, this forces a fresh read of the file.
importlib.reload(pipeline)

# Run the whole bronze-to-silver step on the raw transactions. The result is a DataFrame.
silver = pipeline.build_silver(bronze_tx)

# Print how many rows silver has, and how many bronze had.
# In the f-string, {x:,} formats a number with thousands separators (e.g. 15126 becomes 15,126).
print(f"silver: {len(silver):,} rows (from {len(bronze_tx):,} bronze rows)")


# Define a helper that reports how many rows break a rule and what the pipeline would do.
# name: a label for the check.
# bad_rows: the rows that failed the check.
# action: what to do when rows fail ("fail", "quarantine" or "flag").
def run_check(name, bad_rows, action):
    # A docstring: a description of what the function is for.
    """Report how many rows break a rule, and what the pipeline would do about it."""

    # A dictionary mapping each allowed action to the message printed when rows fail.
    outcomes = {"fail": "PIPELINE STOPS", "quarantine": "rows set aside", "flag": "rows kept and flagged"}

    # If the action isn't one of the three keys (for example "___", the blank
    # students start with), print a prompt to choose one...
    if action not in outcomes:
        # {name:<45} pads the name to 45 characters, left-aligned, so columns line up.
        print(f"{name:<45} choose an action: fail, quarantine or flag")
        # ...and leave the function early. A bare return gives back None.
        return

    # Count the failing rows.
    n = len(bad_rows)

    # Print the name, the count right-aligned in 6 characters ({n:>6}), and then
    # "ok" if there were no failures, otherwise the message for the chosen action.
    print(f"{name:<45} {n:>6} rows  {'ok' if n == 0 else outcomes[action]}")

silver: 15,126 rows (from 28,829 bronze rows)


**Check 1 is a worked example.** In each of the cells below, replace `YOUR_CONDITION_HERE` with a condition that selects the **bad** rows, and replace `___` with an `action` (remember this can be `fail`, `quarantine` or `flag`). Do **checks 2 and 3 first**; checks 4 and 5 are the stretch. Be ready to justify each choice in one sentence.

In [ ]:
# Check 1 (worked example): txn_id is unique
bad = silver[silver["txn_id"].duplicated(keep=False)]
run_check("txn_id is unique", bad, action="fail")

txn_id is unique                                   0 rows  ok


In [ ]:
# Check 2: currency is one of the accepted values (GBP, EUR, USD, CHF)
ACCEPTED = ["GBP", "EUR", "USD", "CHF"]
bad = silver[YOUR_CONDITION_HERE]
run_check("currency is an accepted value", bad, action="___")

In [ ]:
# Check 2: currency is one of the accepted values (GBP, EUR, USD, CHF)
ACCEPTED = ["GBP", "EUR", "USD", "CHF"]
bad = silver[~silver["currency"].isin(ACCEPTED)]
run_check("currency is an accepted value", bad, action="fail")

currency is an accepted value                      0 rows  ok


In [ ]:
# Check 3 (stretch): every card_id exists in the customer master (a 'relationships' test)
bad = silver[~silver["card_id"].isin(bronze_cu["card_id"])]
run_check("card_id exists in customer master", bad, action="flag")

card_id exists in customer master                150 rows  rows kept and flagged


**Justify your choices** (one sentence each). Think about what is lost if the pipeline stops, what is lost if bad rows slip through, and who would need to look at quarantined rows.

1. Currency:
2. Sign rule:
3. Unknown cards:

**Pair discussion (2 minutes):** your unit tests passed on tiny hand-made tables. Which of the five data checks would catch a problem that no unit test of yours would have caught, and why?

---

# Part 2. Version control with git

There is no model answer as there are no questions.